# pythscribe — scalar `@wasm` kernels **in the browser tab** with `client_side(...)` (v0.2.6 fix A)

`browser_wasm_demos.ipynb` showed that Gradio's built-in **`js=` event hook** runs a compiled `@wasm`
kernel client-side with **no custom component and no server round-trip** (~10 ms per event, most of it
Gradio's own frontend overhead; the kernel itself is ~0.1 ms) — but every tab needed ~10 lines of
hand-written JavaScript (import the bundle, parse the card number into digits, encode the text, format
the result). This notebook folds the same **four scalar use cases** onto the new helper,
**`pythscribe.gradio.client_side`**, so an app author writes **no JS at all**:

```python
from pythscribe.gradio import client_side, Arg

cs = client_side({"filter": count_above, "luhn": luhn_ok, "pii": pii_scan, "loan": monthly_payment})
demo.load(None, None, None, js=cs.loader_js)                       # ONE hook binds every kernel's own .wasm
card.change(None, [card], [out, status], js=cs.call_js([Arg.digits], kernel="luhn",
            format="(v, i) => v === 0 ? 'VALID card (Luhn)' : 'INVALID (fails Luhn)'", status=True))
```

`Arg`s map the handler's Gradio inputs onto the kernel's parameters, in declaration order — `Arg.float`,
`Arg.int` (a non-integer is **refused**, never truncated), `Arg.digits`, `Arg.utf8_bytes`, `Arg.codepoints`,
`Arg.floats` / `Arg.ints`, `Arg.const(value)` (embedded once, type-checked against the parameter), and
`Arg.js("v => ...")` as the raw escape hatch. The built-in transforms are implemented **once**, in the
shipped `browser_scalar.js`; the handler runs the kernel's own `.wasm` through the pythscribe FFI shim
(`ffi.call(..., {returnType: "float"})`), which **refuses** rather than re-running on a JS twin, so what
ran is always the WASM export. The contract is checked at **app-build time**: a kernel must return
`float` (M0's one crossed return type — the same authority `dispatch` uses), must not write into a list
parameter (nothing is read back on this path), and must have a usable artifact (no server fallback, ever).

**What is and is not claimed.** The *computation* never leaves the tab: per event the server does
nothing (the headless run below asserts 0 compute requests and a 0 delta on every kernel's server-side
counter). The *inputs* are ordinary Gradio component values — a Textbox / Slider value is component state
Gradio may hold (its initial value comes from the server, and any other event listing the component as an
input sends it) — so "the input never reaches the server" is **not** claimed.

Run in the **PythScribe (Gradio)** kernel (`pyths-gradio`), after `python -m pythscribe.build examples/wasm-use-cases/kernels.py`.

## The app — four use cases, one `call_js(...)` each

Two tabs: **In-tab (client-side)** — the client-side filter (`count_above` over 2000 embedded values,
`Arg.const(DATA), Arg.float`), the Luhn card validator (`Arg.digits`), the on-device PII scan
(`Arg.utf8_bytes, Arg.const(9)`), and the loan calculator (wired the recommended **named** way, each parameter bound to its
component by name); and
**Server round-trip** — the *same* kernels wired through Python `fn`s (one round-trip per event; the
server's counters advance). The second tab is the measurement's **paired negative control**: driven through
the same headless harness it must *fail* the client-side assertions (requests > 0, counters > 0 per kernel),
so a silent fall-back to the server could not pass the gate.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))   # examples/wasm-use-cases
from browser_scalar_client_app import build_demo, KERNELS
from pythscribe import binding_of
from pythscribe.gradio import client_side, Arg

for name, fn in KERNELS.items():
    b = binding_of(fn)
    print(f"{name:7s} -> {b.name:16s} {', '.join(f'{n}: {t}' for n, t in b.params):45s} -> {b.return_type}  artifact={b.artifact_status}")

cs = client_side(KERNELS)   # every kernel's contract + artifact is validated HERE (no fallback)
print()
print("the emitted handler for the card validator:")
print(cs.call_js([Arg.digits], kernel="luhn", format="v => v === 0 ? 'VALID' : 'INVALID'"))

demo = build_demo()
demo.launch(prevent_thread_lock=True, inline=True, quiet=True, server_name="127.0.0.1")

C:\Users\DELL\anaconda3\envs\pyths-gradio\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


filter  -> count_above      xs: list[float], thr: float                   -> float  artifact=resolved
luhn    -> luhn_ok          digits: list[int]                             -> float  artifact=resolved
pii     -> pii_scan         text: list[int], min_run: int                 -> float  artifact=resolved
loan    -> monthly_payment  principal: float, annual_rate_pct: float, months: int -> float  artifact=resolved

the emitted handler for the card validator:
(...a) => {
  const c = window.pythscribeScalar;
  if (!(c && c.ready)) {
    const m = c && c.loadError ? 'in-tab @wasm client failed to load (NO fallback): ' + c.loadError : 'loading the in-tab @wasm client...';
    return false ? [m, m] : m;
  }
  return c.invoke("luhn", a, [{t: "digits"}], (v => v === 0 ? 'VALID' : 'INVALID'), false);
}


## Headless measurement — client-side, bit-for-bit, and the latency contrast

`browser_scalar_probe.py` (the same driver `tests/pythscribe/test_browser_scalar_client_e2e.py` asserts on)
launches the app as a subprocess and drives it in a headless Chromium:

* **In-tab tab:** 10 filter-slider settings, 6 card numbers (valid, invalid, Amex, empty), 4 texts (one
  with non-ASCII characters — the UTF-8 twin matters), 5 loan terms and 2 rates (one of them 0 %, the
  kernel's other branch). It counts every request (and WebSocket) the page makes (`compute_requests` =
  anything under `/gradio_api/` that is not a static `file=` fetch) and reads the server-side kernel
  counters (`python_calls + server_calls`, per kernel) before/after. For every interaction it records the
  float the tab computed (and its IEEE-754 **bits**), the args the `Arg` transforms produced, and the text
  actually **displayed**, and compares each with the kernel's **CPython body** run on independent Python
  twins of the transforms — bit-for-bit.
* **Server tab:** the same 10 filter moves, then one card, one text and one loan move through the Python
  `fn`s — the negative control (asserted **per kernel**) and the latency baseline.

(Runs as a subprocess: Windows' Proactor loop lets Playwright spawn Chromium; Jupyter's cannot.)

In [2]:
import json, subprocess, sys
p = subprocess.run([sys.executable, "browser_scalar_probe.py"], capture_output=True, text=True, encoding="utf-8", cwd=str(Path.cwd()), timeout=900)
line = next((l for l in p.stdout.splitlines() if l.startswith("RESULT_JSON:")), None)
if line is None:
    print("probe failed:\n", (p.stderr or p.stdout)[-2000:])
else:
    r = json.loads(line[len("RESULT_JSON:"):])
    c, s = r["client"], r["server"]
    rows = [
        {"path": "in-tab (client_side + js=)", "median event->result ms": round(c["median_ms"], 1),
         "kernel ms (filter, 2000 values)": round(c["kernel_median_ms"], 2), "compute requests": len(c["compute_requests"]),
         "server kernel calls": c["counters_delta"], "server work per event": "none"},
        {"path": "server round-trip (same kernels)", "median event->result ms": round(s["median_ms"], 1),
         "kernel ms (filter, 2000 values)": None, "compute requests": len(s["compute_requests"]),
         "server kernel calls": s["counters_delta"], "server work per event": "one round-trip + the kernel"},
    ]
    fid = r["fidelity"]
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(rows).set_index("path"))
        display(pd.DataFrame([{"interaction": k, "kernel": v["kernel"], "in-tab value": v["value"], "CPython ref": v["ref"],
                               "bits ==": v["bits_equal"], "displayed ==": v["shown_equal"], "args ==": v["args_equal"],
                               "displayed": v["shown"]} for k, v in fid.items()]).set_index("interaction"))
    except ImportError:
        for row in rows: print(row)
        for k, v in fid.items(): print(k, v)
    print(f"per-kernel server counter delta, in-tab: {c['counters_delta_per_kernel']}  |  server tab: {s['counters_delta_per_kernel']}")
    print(f".wasm fetched {len(c['wasm_requests'])}x (once per kernel); static file fetches {len(c['file_requests'])}; layout {c['layout']}; kernel calls in-tab {c['calls']}")
    print("in-tab status:", c["status"])
    print("server status:", s["status"])
    assert c["compute_requests"] == [] and all(v == 0 for v in c["counters_delta_per_kernel"].values()), "the in-tab path touched the server"
    assert len(s["compute_requests"]) > 0 and all(v > 0 for v in s["counters_delta_per_kernel"].values()), "the negative control did not trip"
    assert r["all_bit_for_bit"], {k: v for k, v in fid.items() if not (v["bits_equal"] and v["shown_equal"] and v["args_equal"])}
    print(f"ALL {r['n_interactions']} interactions bit-for-bit == CPython (float bits, transformed args AND the displayed text); "
          f"in-tab {c['median_ms']:.1f} ms vs server {s['median_ms']:.1f} ms")

,median event->result ms,"kernel ms (filter, 2000 values)",compute requests,server kernel calls,server work per event
path,,,,,
in-tab (client_side + js=),8.9,0.0,0,0,none
server round-trip (same kernels),61.1,NaN,30,15,one round-trip + the kernel


,kernel,in-tab value,CPython ref,bits ==,displayed ==,args ==,displayed
interaction,,,,,,,
filter@0.13,count_above,1717.000000,1717.000000,True,True,True,1717 of 2000 above 0.13
filter@0.42,count_above,1129.000000,1129.000000,True,True,True,1129 of 2000 above 0.42
filter@0.77,count_above,480.000000,480.000000,True,True,True,480 of 2000 above 0.77
filter@0.05,count_above,1880.000000,1880.000000,True,True,True,1880 of 2000 above 0.05
filter@0.9,count_above,203.000000,203.000000,True,True,True,203 of 2000 above 0.90
filter@0.5,count_above,970.000000,970.000000,True,True,True,970 of 2000 above 0.50
filter@0.33,count_above,1297.000000,1297.000000,True,True,True,1297 of 2000 above 0.33
filter@0.61,count_above,774.000000,774.000000,True,True,True,774 of 2000 above 0.61
filter@0.25,count_above,1460.000000,1460.000000,True,True,True,1460 of 2000 above 0.25


per-kernel server counter delta, in-tab: {'filter': 0, 'luhn': 0, 'pii': 0, 'loan': 0}  |  server tab: {'filter': 12, 'luhn': 1, 'pii': 1, 'loan': 1}
.wasm fetched 4x (once per kernel); static file fetches 4; layout pyths-0.2.4-list-v1; kernel calls in-tab 32
in-tab status: loan: 0.00 ms @wasm in-tab (call 32), 0 round-trips, layout pyths-0.2.4-list-v1
server status: filter: 0.89 ms on the SERVER (mode=server); 1 round-trip; call #12; python_calls+=0 server_calls+=1
ALL 27 interactions bit-for-bit == CPython (float bits, transformed args AND the displayed text); in-tab 8.9 ms vs server 61.1 ms


## Reading the numbers

* **Client-side, verified two ways:** 0 compute requests from the page during the interaction (only each
  kernel's `.wasm`, fetched once, both static), and a 0 delta on every kernel's server-side counter — while
  the server tab, through the same harness, produces both for every kernel (the control trips). Not claimed:
  that the input never reaches the server — a Textbox / Slider value is Gradio component state.
* **Bit-for-bit:** every in-tab float equals the CPython kernel run on independent transform twins (IEEE-754
  bits, so `-0.0`/rounding cannot hide), the args the built-in transforms produced equal the twins', and the
  text the page actually displays equals the expected formatting — for 27 (kernel, input) pairs including
  the empty card, non-ASCII text (`Arg.utf8_bytes`), and the loan kernel's 0 % branch. A browserless
  differential (`test_client_side.py`) evaluates the *emitted* handler under Node with the real shim + `.wasm`
  on random inputs against the same CPython bodies, and pins every refusal (`Arg.int` on `12.5`, an
  empty multiselect `[]` or a hex string `"0x10"` — refused, never coerced to `0`/`16` — a wrong
  **count** of `inputs=[...]`, a non-numeric Textbox) as a loud `error (in-tab @wasm, NO fallback)`.
  What a positional `inputs=[...]` cannot catch is a wrong **order** of the right length (a shifted
  call): for that use the **named** `call_js({param: (Arg, component)})` form, which returns
  `(js, inputs)` already in signature order, so a component can never be bound to the wrong parameter.
* **Latency:** the in-tab median is Gradio's own client-side event path plus a ~0.1 ms kernel; the server
  median is the same kernel plus the queue round-trip. The kernel work is trivial here on purpose — the win
  is the round-trip and the deployment shape (no server compute, no custom component), not FLOPs.

**Reuse** — any `@wasm` kernel with `int | float | bool | list[int] | list[float]` parameters and a `float`
return that writes no list:

```python
from pythscribe.gradio import client_side, Arg
cs = client_side({"luhn": luhn_ok, "loan": monthly_payment})
demo.load(None, None, None, js=cs.loader_js)
card.change(None, [card], [out], js=cs.call_js([Arg.digits], kernel="luhn", format="v => v === 0 ? 'VALID' : 'INVALID'"))
# named form (recommended) for a multi-arg kernel: bind each parameter BY NAME -> no positional shift
loan_js, loan_inputs = cs.call_js({"principal": (Arg.float, principal), "annual_rate_pct": (Arg.float, rate),
                                   "months": (Arg.int, months)}, kernel="loan", status=True)
months.change(None, loan_inputs, [out, status], js=loan_js)
```

**Value-domain contract.** The bit-for-bit agreement holds for inputs *within* each kernel's valid domain (here: `Arg.digits` keeps Luhn's inputs 0-9). Outside it the runtimes diverge in a known, one-directional way — pyths' WASM runtime **refuses** i64 overflow / out-of-range access (it traps, surfaced as a loud `error (in-tab @wasm, NO fallback)`) where CPython's big-ints / list bounds might return a value; the one case that would be *silent* is float division by zero (`/` yields ±inf, no trap), which `monthly_payment` guards explicitly. pyths never silently returns a wrong value.

A stub is published before the client finishes loading (an early event shows `loading…`, never a
TypeError); a load failure or a refused input is surfaced in the output — nothing falls back to the server.
Capability limits, stated: `int`/`bool` returns and kernels with scratch/out lists (e.g. `dtw_distance`'s
`prev`/`cur`) are refused on this path (use the component / the image loader); a `gr.Number` output wants
`format=None`, a `gr.Textbox` takes either.